# HFO Evaluation on Real-Time sEEG Data using an snn-torch-trained Network (Fixed-Precision)

## Check WD (change if necessary) and file loading

In [1]:
# Show current directory
import os
curr_dir = os.getcwd()
print(curr_dir)

# Check if the current WD is the file location
if "/thesis-lava/src/nir" not in os.getcwd():
    # Set working directory to this file location
    file_location = f"{os.getcwd()}/thesis-lava/src/nir"
    print("File Location: ", file_location)

    # Change the current working Directory
    os.chdir(file_location)

    # New Working Directory
    print("New Working Directory: ", os.getcwd())

/home/monkin/Desktop/feup/thesis
File Location:  /home/monkin/Desktop/feup/thesis/thesis-lava/src/nir
New Working Directory:  /home/monkin/Desktop/feup/thesis/thesis-lava/src/nir


### Add Parent Directory to Path
Need to add it to access `utils` without adding it to PATH

In [2]:
# Add grandparent directory to path (To acess sntt_utils)
import sys

current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, os.pardir))
# Add the grandparent directory to the system path
# grandparent_dir = os.path.abspath(os.path.join(current_dir, os.pardir, os.pardir))
# sys.path.append(parent_dir)
sys.path.insert(0, parent_dir)


print(sys.path)

['/home/monkin/Desktop/feup/thesis/thesis-lava/src', '/home/monkin/Desktop/feup/thesis', '/home/monkin/Desktop/feup/ncn-lava/src', '/home/monkin/Desktop/feup/thesis/thesis-lava/src', '/home/monkin/Desktop/feup/thesis/lava-dl/src', '/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/home/monkin/Desktop/feup/thesis/.venv/lib/python3.10/site-packages', '/home/monkin/Desktop/feup/thesis/lava-dl', '/home/monkin/Desktop/feup/thesis/.venv/src/lava/src', '/home/monkin/Desktop/feup/thesis/.venv/src/lava']


## Load the Network from the `NIR` file

In [3]:
from utils.hfo import band_to_file_name, MarkerType, BaselineAlgorithm

# ----- Frequency Band Parameters -----
# Declare if using ripples, fast ripples, or both
chosen_band = MarkerType.FAST_RIPPLE     # RIPPLE, FAST_RIPPLE, or BOTH
# Specify the chosen Baseline Algorithm
chosen_baseline_alg_suffix = BaselineAlgorithm.MEDIAN # BaselineAlgorithm.EIGHTY_PERC    # BaselineAlgorithm.Q3
# Get the suffix for the chosen frequency band
BAND_FILENAME = band_to_file_name(chosen_band)

In [5]:
import numpy as np
import nir
import matplotlib.pyplot as plt

# Load the Trained Network from the NIR File
NIR_FILENAME = f"{BAND_FILENAME}_{chosen_baseline_alg_suffix}_trained_net.nir"

nir_network = nir.read(f"models/{NIR_FILENAME}")

### Analyze individual Layers of the imported Network

In [6]:
nirLIF1 = nir_network.nodes["lif1"]
nirLIF2 = nir_network.nodes["lif2"]
nirLIFOut = nir_network.nodes["lif_out"]
print(nirLIF1)

CubaLIF(tau_syn=array([0.00027337, 0.00020684, 0.00029694, 0.0001266 , 0.00028942,
       0.00017805, 0.00018086, 0.00020547, 0.00024664, 0.00011793,
       0.00029177, 0.00020036, 0.00029843, 0.00025447, 0.00028037,
       0.0001865 , 0.00017841, 0.0019396 , 0.00040969, 0.00037363,
       0.00015701, 0.00015839, 0.00016286, 0.00012814], dtype=float32), tau_mem=array([0.00011864, 0.00023361, 0.00015698, 0.00017259, 0.00034534,
       0.00014197, 0.00015067, 0.00051802, 0.00033085, 0.00021419,
       0.00019028, 0.00016002, 0.00017566, 0.00015763, 0.00017741,
       0.00020649, 0.00012887, 0.00017213, 0.00013625, 0.00027133,
       0.00017605, 0.00081964, 0.00023196, 0.00011435], dtype=float32), r=array([1.1863538, 2.3360846, 1.5698305, 1.7258911, 3.4533699, 1.4196671,
       1.5067121, 5.1802025, 3.3085303, 2.1418586, 1.9027638, 1.6001774,
       1.7566377, 1.5762817, 1.7740619, 2.0649114, 1.2887096, 1.7213036,
       1.3625021, 2.7133477, 1.7605472, 8.196443 , 2.3196158, 1.1435146],
 

In [7]:
# Calculate the dv and du parameters from tau_mem and tau_syn
lif1_dv = np.array(list(map(lambda x: (1e-4 / x), nirLIF1.tau_mem)))
lif1_du = np.array(list(map(lambda x: (1e-4 / x), nirLIF1.tau_syn)))

lif2_dv = np.array(list(map(lambda x: (1e-4 / x), nirLIF2.tau_mem)))
lif2_du = np.array(list(map(lambda x: (1e-4 / x), nirLIF2.tau_syn)))

lif_out_dv = (1e-4 / nirLIFOut.tau_mem)
lif_out_du = (1e-4 / nirLIFOut.tau_syn)

In [9]:
nirLIFOut

CubaLIF(tau_syn=np.float32(0.00025984697), tau_mem=np.float32(0.00025984686), r=np.float32(2.5984685), v_leak=np.float32(0.0), v_threshold=np.float32(1.0), w_in=np.float32(2.5984697), input_type={'input': array([], dtype=float64)}, output_type={'output': array([], dtype=float64)}, metadata={}, num_neurons=np.int64(1))

In [10]:
# Print the network summary
print(f"NIR Network Info: Nº Nodes: {len(nir_network.nodes)} | Nº Edges: {len(nir_network.edges)}")

print("Nodes: ", nir_network.nodes)
print(f"\nLIF Nodes dv and du: LIF1: dv: {lif1_dv} | du: {lif1_du}\nLIF2: dv: {lif2_dv} | du: {lif2_du}\nLIFOut: dv: {lif_out_dv} | du: {lif_out_du}")

print("\nEdges: ", nir_network.edges)

NIR Network Info: Nº Nodes: 8 | Nº Edges: 7
Nodes:  {'fc3': Linear(weight=array([[-1.11489333e-01, -2.68412918e-01, -3.24451417e-01,
         6.16860867e-01, -3.27180773e-01, -2.12263197e-01,
        -1.17689617e-01, -2.33668327e-01,  5.08770943e-01,
        -4.38423663e-01,  4.84119266e-01,  5.60897887e-01,
        -3.88121665e-01,  6.27892852e-01,  2.49766069e-03,
         4.52961385e-01, -9.30073857e-02,  6.34308040e-01,
         6.06913209e-01, -4.26406972e-02,  9.06697035e-01,
         5.93553901e-01, -3.50169949e-02, -8.73716474e-02],
       [ 4.99247432e-01, -1.59530684e-01, -4.45856247e-03,
         7.08272755e-01,  4.83858675e-01,  4.60404038e-01,
         8.41623485e-01,  4.03165221e-01, -4.95698377e-02,
        -1.76092297e-01, -1.59834176e-01,  5.57745457e-01,
         4.06628132e-01,  5.90626299e-01,  4.01989132e-01,
         4.52161670e-01,  5.40686071e-01,  4.61366415e-01,
         5.68166554e-01, -1.86424062e-01, -4.92532272e-03,
         5.76662421e-01, -6.11825660e-02

In [11]:
from typing import TypedDict

class LifNode(TypedDict):
    tau_syn: np.float32
    tau_mem: np.float32
    r: np.float32
    v_leak: np.float32
    v_threshold: np.float32
    w_in: np.float32
    input_type: dict[str, list[np.float64]]
    output_type: dict[str, list[np.float64]]
    metadata: dict
    num_neurons: np.int64

nirLifOut: LifNode = nir_network.nodes["lif_out"]

## Transform the NIR Representation to a Lava Network

In [12]:
from nir_to_lava_edit import ImportConfig, LavaLibrary, import_from_nir
# from nir_to_lava import ImportConfig, LavaLibrary, import_from_nir

nir_dt = 1e-4
config = ImportConfig(
    dt=nir_dt, fixed_pt=False, on_chip=False, library_preference=LavaLibrary.Lava,
)

## Read the Input Data and the Ground Truth

In [13]:
IS_CLINICAL = False
PATIENT_LABEL = "csl"
INPUT_TYPE = "clinical" if IS_CLINICAL else "synthetic"

In [14]:
from utils.io import preview_np_array

# Define the channel suffix (Channel range indicates a certain Brain Region & SNR level)
CH_SUFFIX = "ch90-119"  # "ch90-119"

# Get the suffix for the chosen frequency band
BAND_FILENAME = band_to_file_name(chosen_band)

# Define the INPUT Folder
INPUT_FOLDER = f"input_data/{BAND_FILENAME}_{chosen_baseline_alg_suffix}_{CH_SUFFIX}"

In [15]:
# Load the input data
up_spikes = np.load(f"{INPUT_FOLDER}/up_spike_train.npy")
down_spikes = np.load(f"{INPUT_FOLDER}/down_spike_train.npy")
# Load the Ground Truth
gt_data = np.load(f"{INPUT_FOLDER}/gt_data.npy")

# Print the shape of the loaded data
print("Up Spikes Shape: ", up_spikes.shape)
print("Down Spikes Shape: ", down_spikes.shape)
print("GT Data Shape: ", gt_data.shape)

Up Spikes Shape:  (175135,)
Down Spikes Shape:  (175019,)
GT Data Shape:  (720,)


In [16]:
# Define np.array containing the timestep of GT events
gt_times = np.array([int(round(gt_elem[1])) for gt_elem in gt_data])

# Preview the data
preview_np_array(gt_times, "gt_times", edge_items=2)

gt_times Shape: (720,).
Preview: [   4219    6967 ... 3595019 3599000]


In [17]:
# See the time of the first and last UP / DN spikes
first_up, last_up = up_spikes[0], up_spikes[-1]
first_dn, last_dn = down_spikes[0], down_spikes[-1]

print(f"First Up Spike: {first_up} | Last Up Spike: {last_up}")
print(f"First Down Spike: {first_dn} | Last Down Spike: {last_dn}")

First Up Spike: 206.54296875 | Last Up Spike: 3599996.58203125
First Down Spike: 205.56640625 | Last Down Spike: 3599997.55859375


## Define Important Parameters for the Evaluation

In [18]:
from utils.input import INPUT_DURATION, NUM_CH_PER_SNR 

# Simulation Time Parameters
num_steps_per_ch = int(INPUT_DURATION)     # int(INPUT_DURATION * NUM_CH_PER_SNR)    # Number of steps to run the simulation   # TODO: CAREFUL WITH THIS DURATION (INPUT_DURATION IS THE DURATION OF THE SYNTHETIC DATA)
num_channels = NUM_CH_PER_SNR
init_offset = 0 # 900 # 33400      #   
virtual_time_step_interval = 1  # dt = 1 ms

print(f"Num Steps: {num_steps_per_ch}")

Num Steps: 120000


### Split the Input Data and Ground Truth by the number of channels.

In [19]:
'''
Split the UP and DOWN Spikes by the number of channels.
Each Channel has NUM_STEPS_PER_CH timesteps
'''
up_spikes_per_ch, down_spikes_per_ch = np.ndarray(shape=(NUM_CH_PER_SNR,), dtype=object), np.ndarray(shape=(NUM_CH_PER_SNR,), dtype=object)
gt_times_per_ch = np.ndarray(shape=(NUM_CH_PER_SNR,), dtype=object)

for up_spike in up_spikes:
    ch_idx = int(up_spike // INPUT_DURATION)
    if up_spikes_per_ch[ch_idx] is None:
        # Remove the first None element
        up_spikes_per_ch[ch_idx] = np.array([up_spike - (ch_idx * INPUT_DURATION)], dtype=np.float64)
    else:
        up_spikes_per_ch[ch_idx] = np.append(up_spikes_per_ch[ch_idx], up_spike - (ch_idx * INPUT_DURATION))
for down_spike in down_spikes:
    ch_idx = int(down_spike // INPUT_DURATION)
    if down_spikes_per_ch[ch_idx] is None:
        # Remove the first None element
        down_spikes_per_ch[ch_idx] = np.array([down_spike - (ch_idx * INPUT_DURATION)], dtype=np.float64)
    else:
        down_spikes_per_ch[ch_idx] = np.append(down_spikes_per_ch[ch_idx], down_spike - (ch_idx * INPUT_DURATION))
for gt_time in gt_times:
    ch_idx = int(gt_time // INPUT_DURATION)
    if gt_times_per_ch[ch_idx] is None:
        # Remove the first None element
        gt_times_per_ch[ch_idx] = np.array([gt_time - (ch_idx * INPUT_DURATION)], dtype=np.float64)
    else:
        gt_times_per_ch[ch_idx] = np.append(gt_times_per_ch[ch_idx], gt_time - (ch_idx * INPUT_DURATION))

# Print the shape of the loaded data
print("Up Spikes Per Channel Shape: ", up_spikes_per_ch.shape)
print("Down Spikes Per Channel Shape: ", down_spikes_per_ch.shape)
print("GT Times Per Channel Shape: ", gt_times_per_ch.shape)

Up Spikes Per Channel Shape:  (30,)
Down Spikes Per Channel Shape:  (30,)
GT Times Per Channel Shape:  (30,)


### Check if the Input data and Ground Truth is well-distributed among the channels

In [20]:
# Check if UP spikes are more or less distributed by the INPUT CHANNELS
for ch_idx in range(NUM_CH_PER_SNR):
    ch_up_spikes = up_spikes_per_ch[ch_idx].size
    ch_down_spikes = down_spikes_per_ch[ch_idx].size
    ch_gt_times = gt_times_per_ch[ch_idx].size
    print(f"""Nº UP / DN Spikes for Channel {ch_idx}: {ch_up_spikes}({round((ch_up_spikes / up_spikes.size)*100, 2)}%) / {ch_down_spikes}({round((ch_down_spikes / down_spikes.size)*100, 2)}%)
          | Nº GT Times: {ch_gt_times}({round((ch_gt_times / gt_times.size)*100, 2)}%)""")

Nº UP / DN Spikes for Channel 0: 5513(3.15%) / 5525(3.16%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 1: 6147(3.51%) / 6144(3.51%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 2: 6049(3.45%) / 6036(3.45%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 3: 5921(3.38%) / 5912(3.38%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 4: 5632(3.22%) / 5632(3.22%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 5: 5901(3.37%) / 5882(3.36%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 6: 5654(3.23%) / 5640(3.22%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 7: 6042(3.45%) / 6040(3.45%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 8: 5679(3.24%) / 5669(3.24%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 9: 5703(3.26%) / 5698(3.26%)
          | Nº GT Times: 24(3.33%)
Nº UP / DN Spikes for Channel 10: 5849(3.34%) / 5843(3.34%)


As we can see, the spikes were well distributed.

Now, we can perform the remaining steps inside a loop to detect the HFOs 1 sEEG Channel at a time to reduce the memory usage.

# HFO Evaluation

In [ ]:
from detect_hfo import evaluate_hfo_detector, HFOEvalResults

# Store the results of the HFO Evaluation for each sEEG Channel
prediction_results: list[HFOEvalResults] = []
for ch_idx in range(NUM_CH_PER_SNR):
    # Get the Input Data for the current channel
    curr_spk_evts = np.array(
        [up_spikes_per_ch[ch_idx], down_spikes_per_ch[ch_idx]], dtype=object)
    
    # Get the GT Times for the current channel
    curr_gt_times = gt_times_per_ch[ch_idx]
    
    EVAL_LABEL = f"CH Index: {ch_idx}"

    USE_REFRAC = False  # Whether to use Refractory Neurons in the Output Layer

    # Run the HFO Evaluation
    print(f"\n\n--- HFO Evaluation - {EVAL_LABEL} ---")
    curr_eval_results = evaluate_hfo_detector(
        nir_network, config, chosen_band, USE_REFRAC, curr_spk_evts, curr_gt_times,
        num_steps_per_ch, EVAL_LABEL, verbose=False
    )

    prediction_results.append(curr_eval_results)



--- HFO Evaluation - CH Index: 0 ---


/home/monkin/Desktop/feup/thesis/thesis-lava/src/lava/magma/compiler/compiler_graphs.py:895: UserWarning: Cannot import module '<module 'snn' from '/home/monkin/Desktop/feup/thesis/thesis-lava/src/utils/snn.py'>' when searching ProcessModels for Process 'SpikeEventGen_v2'.
  warnings.warn(


Time step: 10000
Time step: 20000
Time step: 30000
Time step: 40000
Time step: 50000
Time step: 60000
Time step: 70000
Time step: 80000
Time step: 90000
Time step: 100000
Time step: 110000
Time step: 120000
Confusion Matrix:
|TP: 21 | FP: 4|
|FN: 3  | TN: 0|
Recall (True Positive Rate): 87.50 %
Precision (TP / (TP + FP)): 84.00 %
F1 Score (Combines Precision & Recall): 85.71 %
Total Predictions: 28


--- HFO Evaluation - CH Index: 1 ---
Time step: 10000
Time step: 20000
Time step: 30000
Time step: 40000
Time step: 50000
Time step: 60000
Time step: 70000
Time step: 80000
Time step: 90000
Time step: 100000
Time step: 110000
Time step: 120000
Confusion Matrix:
|TP: 19 | FP: 8|
|FN: 5  | TN: 0|
Recall (True Positive Rate): 79.17 %
Precision (TP / (TP + FP)): 70.37 %
F1 Score (Combines Precision & Recall): 74.51 %
Total Predictions: 32


--- HFO Evaluation - CH Index: 2 ---
Time step: 10000
Time step: 20000
Time step: 30000
Time step: 40000
Time step: 50000
Time step: 60000
Time step: 70000

Process SystemProcess-85:
Process SystemProcess-87:
Process SystemProcess-83:
Process SystemProcess-84:
Process SystemProcess-79:
Process SystemProcess-86:
Process SystemProcess-81:
Process SystemProcess-80:


KeyboardInterrupt: 

Process SystemProcess-82:
Process SystemProcess-88:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/monkin/Desktop/feup/thesis/thesis-lava/src/lava/magma/runtime/message_infrastructure/multiprocessing.py"

  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/monkin/Desktop/feup/thesis/thesis-lava/src/lava/magma/runtime/message_infrastructure/multiprocessing.py", line 68, in run
    mp.Process.run(self)
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/monkin/Desktop/feup/thesis/thesis-lava/src/lava/magma/runtime/message_infrastructure/multiprocessing.py", line 68, in run
    mp.Process.run(self)
  File "/usr/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/usr/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/home/monkin/Desktop/feup/thesis/thesis-lava/src/lava/magma/runtime/message_infrastructure/multiprocessing.py", line 68, in run
    mp.Process.run(self)
  File "/home/monkin/Desktop/feup/thesis/thesis-lava/src/lava/magma/runtime/message_infrastructure/multiproce

## Show the Prediction Results

In [41]:
print(f"prediction_results: {prediction_results}")

prediction_results: [{'label': 'CH Index: 0', 'TP': 21, 'FP': 4, 'FN': 3, 'TN': 0, 'recall': 0.875, 'precision': 0.84, 'f1_score': 0.8571428571428572, 'total_predictions': 28}, {'label': 'CH Index: 1', 'TP': 19, 'FP': 8, 'FN': 5, 'TN': 0, 'recall': 0.7916666666666666, 'precision': 0.7037037037037037, 'f1_score': 0.7450980392156864, 'total_predictions': 32}, {'label': 'CH Index: 2', 'TP': 22, 'FP': 2, 'FN': 2, 'TN': 0, 'recall': 0.9166666666666666, 'precision': 0.9166666666666666, 'f1_score': 0.9166666666666666, 'total_predictions': 26}]


## Join the results from all channels

In [42]:
TP_ALL, FP_ALL, FN_ALL, TN_ALL = 0, 0, 0, 0
for ch_idx in range(len(prediction_results)):
    curr_results = prediction_results[ch_idx]
    TP_ALL += curr_results["TP"]
    FP_ALL += curr_results["FP"]
    FN_ALL += curr_results["FN"]
    TN_ALL += curr_results["TN"]

print(f"|TP: {TP_ALL} | FP: {FP_ALL}|\n|FN: {FN_ALL}  | TN: {TN_ALL}|")

|TP: 62 | FP: 14|
|FN: 10  | TN: 0|


In [43]:
# Calculate the performance metrics (Not including metrics that rely on TN)
recall = TP_ALL / (TP_ALL + FN_ALL) if (TP_ALL + FN_ALL) > 0 else 0
precision = TP_ALL / (TP_ALL + FP_ALL) if (TP_ALL + FP_ALL) > 0 else 0
f1_score = 2 * (precision * recall) / (precision +
                                        recall) if (precision + recall) > 0 else 0
total_predictions = TP_ALL + FP_ALL + FN_ALL
# Output the performance metrics
print(f"Recall (True Positive Rate): {recall*100:.2f} %")
print(f"Precision (TP / (TP + FP)): {precision*100:.2f} %")
print(f"F1 Score (Combines Precision & Recall): {f1_score*100:.2f} %")
print(f"Total Predictions: {total_predictions}")

Recall (True Positive Rate): 86.11 %
Precision (TP / (TP + FP)): 81.58 %
F1 Score (Combines Precision & Recall): 83.78 %
Total Predictions: 86


# Export the results of the Classification to a JSON file
Export the results of the classification to a JSON file. This file will include:
- Frequency Band used (`Ripple`, `Fast Ripple` or `Both`).
- Channels Used.
- `num_steps`
- `Confidence Window` used
- Classification Metrics (`True Positives`, `False Positives`, `False Negatives`, `Precision`, `Recall`, `F1 Score`)

In [ ]:
from utils.hfo import band_to_gt_max_offset, band_to_file_name, MarkerType, BaselineAlgorithm

# Giving PRED_CAUSALITY_WINDOW ms for the network to update its inner state and spike
PRED_CAUSALITY_WINDOW = int(5)
MAX_DETECTION_OFFSET = int(band_to_gt_max_offset(
        chosen_band)) * 1.5 + PRED_CAUSALITY_WINDOW   # in timesteps (ms)

In [ ]:
import json

# Export the results to a JSON file
OUTPUT_FOLDER = f"eval/{INPUT_TYPE}"
# create the output folder if it doesn't exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Create a dictionary with the results
json_results = {
    "freq_band": BAND_FILENAME,
    "channels": CH_SUFFIX,
    "max_detection_offset": MAX_DETECTION_OFFSET,
    "baseline_algorithm": chosen_baseline_alg_suffix,
    "metrics": {
        "true_positive": TP_ALL,
        "false_positive": FP_ALL,
        "false_negative": FN_ALL,
        "total_predictions": total_predictions,
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
    }
}

EXPORT_JSON_FILE = True
if EXPORT_JSON_FILE:
    json_file_name = f"{OUTPUT_FOLDER}/{BAND_FILENAME}_{chosen_baseline_alg_suffix}_{CH_SUFFIX}_results_.json"
    with open(json_file_name, 'w') as f:
        json.dump(json_results, f)